In [1]:
import pandas as pd

# Load dataset (replace with your file path)
df = pd.read_csv("Petrol Dataset June 23 2022 -- Version 2.csv")

# --------------------------
# 1. Fix misaligned Côte d'Ivoire row
# --------------------------
# Find the row where Country is empty and fix it
mask = df['Country'].isna()
if mask.any():
    idx = mask.idxmax()
    # Shift values right and assign correct S#
    df.iloc[idx, 1:] = df.iloc[idx, :-1].values
    df.at[idx, 'S#'] = 100
    df.at[idx, 'Country'] = "Côte d'Ivoire"

# --------------------------
# 2. Clean column names
# --------------------------
df.columns = df.columns.str.strip()  # Remove extra spaces
df = df.rename(columns={'GDP Per Capita ( USD )': 'GDP Per Capita (USD)'})

# --------------------------
# 3. Convert numeric columns (remove commas, %, etc.)
# --------------------------
# Columns that need cleaning
cols_to_clean = [
    'Daily Oil Consumption (Barrels)', 'World Share',
    'GDP Per Capita (USD)', 'Gallons GDP Per Capita Can Buy'
]

for col in cols_to_clean:
    if col == 'World Share':
        df[col] = df[col].str.replace('%', '').astype(float) / 100  # % → decimal
    else:
        df[col] = df[col].astype(str).str.replace(',', '').astype(float)

# Ensure other price columns are numeric
price_cols = ['Yearly Gallons Per Capita', 'Price Per Gallon (USD)',
              'Price Per Liter (USD)', 'Price Per Liter (PKR)',
              'xTimes Yearly Gallons Per Capita Buy']
for col in price_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# --------------------------
# 4. Fix S# column
# --------------------------
df['S#'] = pd.to_numeric(df['S#'], errors='coerce').fillna(100).astype(int)

# --------------------------
# 5. Save cleaned data
# --------------------------
df.to_csv("Cleaned_Petrol_Dataset.csv", index=False)

print("✅ Cleaning complete!")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("\nFirst 5 rows preview:")
print(df.head())


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf4 in position 6823: invalid continuation byte

In [2]:
import pandas as pd

# Load file with correct encoding
df = pd.read_csv(
    "Petrol Dataset June 23 2022 -- Version 2.csv",
    encoding="utf-8",
    encoding_errors="replace"  # Fixes all Unicode issues
)

# 1. Fix misaligned Côte d'Ivoire row
mask = df['Country'].isna()
if mask.any():
    idx = mask.idxmax()
    df.iloc[idx, 1:] = df.iloc[idx, :-1].values
    df.at[idx, 'S#'] = 100
    df.at[idx, 'Country'] = "Côte d'Ivoire"

# 2. Clean column names
df.columns = df.columns.str.strip()
df = df.rename(columns={'GDP Per Capita ( USD )': 'GDP Per Capita (USD)'})

# 3. Convert numeric columns
cols_to_clean = [
    'Daily Oil Consumption (Barrels)', 'World Share',
    'GDP Per Capita (USD)', 'Gallons GDP Per Capita Can Buy'
]

for col in cols_to_clean:
    if col == 'World Share':
        df[col] = df[col].astype(str).str.replace('%', '').astype(float) / 100
    else:
        df[col] = df[col].astype(str).str.replace(',', '').astype(float)

# Convert price/value columns
price_cols = ['Yearly Gallons Per Capita', 'Price Per Gallon (USD)',
              'Price Per Liter (USD)', 'Price Per Liter (PKR)',
              'xTimes Yearly Gallons Per Capita Buy']
for col in price_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 4. Fix S# column
df['S#'] = pd.to_numeric(df['S#'], errors='coerce').fillna(100).astype(int)

# 5. Save cleaned file
df.to_csv("Cleaned_Petrol_Dataset.csv", index=False, encoding="utf-8")

print("✅ Cleaning done! No Unicode errors now.")
print(f"Total rows: {df.shape[0]} | Total columns: {df.shape[1]}")


✅ Cleaning done! No Unicode errors now.
Total rows: 181 | Total columns: 11


In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for clean charts
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# --------------------------
# 1. Load Cleaned Data
# --------------------------
df = pd.read_csv("Cleaned_Petrol_Dataset.csv", encoding="utf-8")

# --------------------------
# 2. Basic Analysis & Insights
# --------------------------
print("===== BASIC DATA SUMMARY =====")
print(f"Total Countries: {df['Country'].nunique()}")
print("\n===== TOP 5 OIL CONSUMERS =====")
print(df.nlargest(5, 'Daily Oil Consumption (Barrels)')[['Country','Daily Oil Consumption (Barrels)']])
print("\n===== TOP 5 MOST EXPENSIVE PETROL (USD/GAL) =====")
print(df.nlargest(5, 'Price Per Gallon (USD)')[['Country','Price Per Gallon (USD)']])
print("\n===== TOP 5 CHEAPEST PETROL (USD/GAL) =====")
print(df.nsmallest(5, 'Price Per Gallon (USD)')[['Country','Price Per Gallon (USD)']])
print("\n===== TOP 5 MOST AFFORDABLE =====")
print(df.nlargest(5, 'Gallons GDP Per Capita Can Buy')[['Country','Gallons GDP Per Capita Can Buy']])
print("\n===== CORRELATION =====")
print(df[['GDP Per Capita (USD)','Price Per Gallon (USD)','Gallons GDP Per Capita Can Buy']].corr())

# --------------------------
# 3. Visualizations
# --------------------------

# --- Chart 1: Top 10 Countries by Daily Oil Consumption ---
top10_consume = df.nlargest(10, 'Daily Oil Consumption (Barrels)')
plt.figure(figsize=(12,6))
sns.barplot(data=top10_consume, x='Daily Oil Consumption (Barrels)', y='Country', palette='Blues_d')
plt.title('Top 10 Countries by Daily Oil Consumption (Barrels)', fontsize=14)
plt.xlabel('Barrels per Day')
plt.tight_layout()
plt.savefig('1_Top10_Oil_Consumption.png', dpi=300)
plt.close()

# --- Chart 2: Fuel Price vs GDP Per Capita ---
plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='GDP Per Capita (USD)', y='Price Per Gallon (USD)', alpha=0.7, color='crimson')
plt.title('Fuel Price vs GDP Per Capita', fontsize=14)
plt.xlabel('GDP Per Capita (USD)')
plt.ylabel('Price Per Gallon (USD)')
plt.tight_layout()
plt.savefig('2_Price_vs_GDP.png', dpi=300)
plt.close()

# --- Chart 3: Top 10 Most Expensive vs Cheapest Petrol ---
top10_cheap = df.nsmallest(10, 'Price Per Gallon (USD)')
top10_expensive = df.nlargest(10, 'Price Per Gallon (USD)')
combined = pd.concat([top10_cheap, top10_expensive])

plt.figure(figsize=(14,6))
sns.barplot(data=combined, x='Country', y='Price Per Gallon (USD)', palette=['green']*10 + ['red']*10)
plt.title('10 Cheapest vs 10 Most Expensive Petrol Prices (USD/Gallon)', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('3_Cheapest_vs_Expensive.png', dpi=300)
plt.close()

# --- Chart 4: Nigeria vs Regional Average ---
nigeria = df[df['Country']=='Nigeria']
africa = df[df['Country'].isin(['Nigeria','Egypt','South Africa','Algeria','Morocco','Kenya','Ghana'])]

plt.figure(figsize=(10,5))
sns.barplot(data=africa, x='Country', y='Price Per Gallon (USD)', palette='viridis')
plt.title('Petrol Price: Nigeria vs Selected African Countries', fontsize=14)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('4_Nigeria_vs_Africa.png', dpi=300)
plt.close()

print("\n✅ All analysis & visualizations saved successfully!")


===== BASIC DATA SUMMARY =====
Total Countries: 181

===== TOP 5 OIL CONSUMERS =====
         Country  Daily Oil Consumption (Barrels)
0  United States                       19687287.0
1          China                       12791553.0
2          India                        4443000.0
3          Japan                        4012877.0
4         Russia                        3631287.0

===== TOP 5 MOST EXPENSIVE PETROL (USD/GAL) =====
         Country  Price Per Gallon (USD)
147  North Korea                   54.89
180        Tonga                   16.20
177         Niue                   11.43
40     Hong Kong                   11.35
58        Norway                   10.22

===== TOP 5 CHEAPEST PETROL (USD/GAL) =====
       Country  Price Per Gallon (USD)
30   Venezuela                    0.08
49       Libya                    0.12
11        Iran                    0.20
125    Brunei                     0.83
69       Syria                    1.08

===== TOP 5 MOST AFFORDABLE =====
    

C:\Users\JOSEPH CALEB\AppData\Local\Temp\ipykernel_11224\2915096874.py:37: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=top10_consume, x='Daily Oil Consumption (Barrels)', y='Country', palette='Blues_d')
C:\Users\JOSEPH CALEB\AppData\Local\Temp\ipykernel_11224\2915096874.py:60: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=combined, x='Country', y='Price Per Gallon (USD)', palette=['green']*10 + ['red']*10)
C:\Users\JOSEPH CALEB\AppData\Local\Temp\ipykernel_11224\2915096874.py:72: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data


✅ All analysis & visualizations saved successfully!
